In [1]:
import pandas as pd
import numpy as np

import re
import string
import joblib
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

In [2]:
import sys

print(sys.executable)

d:\Projects\Codsoft\Movie Genre Classification\.venv\Scripts\python.exe


In [3]:
# Download required NLTK resources

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...


True

In [4]:
train_df = pd.read_csv(
    "../data/train_data.txt",
    sep=" ::: ",
    engine="python",
    header=None,
    names=["ID", "Title", "Genre", "Description"]
)

In [5]:
test_df = pd.read_csv(
    "../data/test_data.txt",
    sep=" ::: ",
    engine="python",
    header=None,
    names=["ID", "Title", "Description"]
)

In [6]:
test_solution = pd.read_csv(
    "../data/test_data_solution.txt",
    sep=" ::: ",
    engine="python",
    header=None,
    names=["ID", "Title", "Genre", "Description"]
)

In [7]:
print("Train Shape :", train_df.shape)
print("Test Shape :", test_df.shape)
print("Test Labels Shape :", test_solution.shape)

Train Shape : (54214, 4)
Test Shape : (54200, 3)
Test Labels Shape : (54200, 4)


In [8]:
# Initialize stopwords and lemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

print("Total Stopwords:", len(stop_words))

Total Stopwords: 198


In [9]:
# Text preprocessing function

def preprocess_text(text):
    
    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenization
    words = text.split()

    # Remove stopwords
    words = [word for word in words if word not in stop_words]

    # Lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [10]:
sample_text = train_df.loc[0, "Description"]

print("Original Text:\n")
print(sample_text)

print("\n" + "="*100 + "\n")

print("Clean Text:\n")
print(preprocess_text(sample_text))

Original Text:

Listening in to a conversation between his doctor and parents, 10-year-old Oscar learns what nobody has the courage to tell him. He only has a few weeks to live. Furious, he refuses to speak to anyone except straight-talking Rose, the lady in pink he meets on the hospital stairs. As Christmas approaches, Rose uses her fantastical experiences as a professional wrestler, her imagination, wit and charm to allow Oscar to live life and love to the full, in the company of his friends Pop Corn, Einstein, Bacon and childhood sweetheart Peggy Blue.


Clean Text:

listening conversation doctor parent yearold oscar learns nobody courage tell week live furious refuse speak anyone except straighttalking rose lady pink meet hospital stair christmas approach rose us fantastical experience professional wrestler imagination wit charm allow oscar live life love full company friend pop corn einstein bacon childhood sweetheart peggy blue


In [11]:
# Apply preprocessing to training and test datasets

train_df["clean_description"] = train_df["Description"].apply(preprocess_text)

test_df["clean_description"] = test_df["Description"].apply(preprocess_text)

print("Preprocessing completed successfully!")

Preprocessing completed successfully!


In [12]:
# Convert text into TF-IDF features

tfidf = TfidfVectorizer(
    max_features=5000
)

X_train = tfidf.fit_transform(train_df["clean_description"])
X_test = tfidf.transform(test_df["clean_description"])

y_train = train_df["Genre"]
y_test = test_solution["Genre"]

In [13]:
print("X_train Shape :", X_train.shape)
print("X_test Shape  :", X_test.shape)

print("y_train Shape :", y_train.shape)
print("y_test Shape  :", y_test.shape)

X_train Shape : (54214, 5000)
X_test Shape  : (54200, 5000)
y_train Shape : (54214,)
y_test Shape  : (54200,)


In [14]:
# Train Multinomial Naive Bayes

nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

print("Multinomial Naive Bayes model trained successfully!")

Multinomial Naive Bayes model trained successfully!


In [15]:
# Predict on test data

nb_predictions = nb_model.predict(X_test)

In [16]:
# Evaluate Naive Bayes model

nb_accuracy = accuracy_score(y_test, nb_predictions)
nb_f1 = f1_score(y_test, nb_predictions, average="macro")

print(f"Accuracy : {nb_accuracy:.4f}")
print(f"Macro F1 Score : {nb_f1:.4f}")

Accuracy : 0.5203
Macro F1 Score : 0.1670


In [17]:
# Classification Report

print(classification_report(y_test, nb_predictions))

d:\Projects\Codsoft\Movie Genre Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

      action       0.55      0.09      0.16      1314
       adult       0.52      0.08      0.14       590
   adventure       0.81      0.09      0.16       775
   animation       0.00      0.00      0.00       498
   biography       0.00      0.00      0.00       264
      comedy       0.52      0.43      0.47      7446
       crime       0.00      0.00      0.00       505
 documentary       0.57      0.87      0.68     13096
       drama       0.46      0.82      0.59     13612
      family       0.67      0.00      0.01       783
     fantasy       0.00      0.00      0.00       322
   game-show       0.98      0.23      0.37       193
     history       0.00      0.00      0.00       243
      horror       0.70      0.33      0.45      2204
       music       0.74      0.14      0.24       731
     musical       0.00      0.00      0.00       276
     mystery       0.00      0.00      0.00       318
        news       0.00    

d:\Projects\Codsoft\Movie Genre Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Projects\Codsoft\Movie Genre Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [18]:
# Train Logistic Regression

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [19]:
# Predict on test data

lr_predictions = lr_model.predict(X_test)

In [20]:
# Evaluate Logistic Regression

lr_accuracy = accuracy_score(y_test, lr_predictions)
lr_f1 = f1_score(y_test, lr_predictions, average="macro")

print(f"Accuracy : {lr_accuracy:.4f}")
print(f"Macro F1 Score : {lr_f1:.4f}")

Accuracy : 0.5856
Macro F1 Score : 0.3016


In [21]:
print(classification_report(y_test, lr_predictions))

d:\Projects\Codsoft\Movie Genre Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Projects\Codsoft\Movie Genre Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

      action       0.46      0.29      0.36      1314
       adult       0.61      0.25      0.36       590
   adventure       0.57      0.17      0.27       775
   animation       0.47      0.07      0.13       498
   biography       0.00      0.00      0.00       264
      comedy       0.53      0.59      0.56      7446
       crime       0.32      0.04      0.07       505
 documentary       0.67      0.85      0.75     13096
       drama       0.55      0.77      0.64     13612
      family       0.51      0.10      0.17       783
     fantasy       0.50      0.05      0.10       322
   game-show       0.84      0.53      0.65       193
     history       0.00      0.00      0.00       243
      horror       0.65      0.58      0.61      2204
       music       0.65      0.45      0.53       731
     musical       0.14      0.01      0.02       276
     mystery       0.33      0.01      0.02       318
        news       0.67    

d:\Projects\Codsoft\Movie Genre Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [22]:
# Train Linear Support Vector Machine

svm_model = LinearSVC(random_state=42)

svm_model.fit(X_train, y_train)

print("Linear SVM model trained successfully!")

Linear SVM model trained successfully!


In [23]:
# Predict on test data

svm_predictions = svm_model.predict(X_test)

In [24]:
# Evaluate Linear SVM

svm_accuracy = accuracy_score(y_test, svm_predictions)
svm_f1 = f1_score(y_test, svm_predictions, average="macro")

print(f"Accuracy : {svm_accuracy:.4f}")
print(f"Macro F1 Score : {svm_f1:.4f}")

Accuracy : 0.5735
Macro F1 Score : 0.3382


In [25]:
print(classification_report(y_test, svm_predictions))

              precision    recall  f1-score   support

      action       0.39      0.32      0.35      1314
       adult       0.54      0.41      0.46       590
   adventure       0.41      0.21      0.28       775
   animation       0.32      0.15      0.20       498
   biography       0.08      0.00      0.01       264
      comedy       0.53      0.57      0.55      7446
       crime       0.20      0.07      0.10       505
 documentary       0.68      0.82      0.74     13096
       drama       0.57      0.71      0.63     13612
      family       0.34      0.15      0.20       783
     fantasy       0.32      0.11      0.16       322
   game-show       0.76      0.62      0.68       193
     history       0.13      0.02      0.03       243
      horror       0.57      0.60      0.58      2204
       music       0.59      0.50      0.54       731
     musical       0.21      0.05      0.08       276
     mystery       0.23      0.06      0.09       318
        news       0.44    

In [26]:
# Compare model performance

comparison = pd.DataFrame({
    "Model": [
        "Multinomial Naive Bayes",
        "Logistic Regression",
        "Linear SVM"
    ],
    "Accuracy": [
        nb_accuracy,
        lr_accuracy,
        svm_accuracy
    ],
    "Macro F1 Score": [
        nb_f1,
        lr_f1,
        svm_f1
    ]
})

comparison = comparison.sort_values(by="Accuracy", ascending=False)

comparison

,Model,Accuracy,Macro F1 Score
1,Logistic Regression,0.585646,0.301616
2,Linear SVM,0.573487,0.338185
0,Multinomial Naive Bayes,0.520332,0.166991


In [27]:
# Save Logistic Regression model and TF-IDF vectorizer

joblib.dump(lr_model, "../models/best_model.joblib")
joblib.dump(tfidf, "../models/tfidf_vectorizer.joblib")

print("Model and TF-IDF Vectorizer saved successfully!")

Model and TF-IDF Vectorizer saved successfully!


In [ ]:
# # Conclusion

# Three machine learning models were trained and evaluated for movie genre classification.

# - Multinomial Naive Bayes served as the baseline model.
# - Logistic Regression achieved the highest overall accuracy.
# - Linear SVM obtained the highest Macro F1 score.

# Based on overall performance, Logistic Regression was selected as the final model for deployment.

# The trained model and TF-IDF vectorizer were saved for use in the Streamlit web application.